In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,RandomizedSearchCV,GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

In [ ]:
data=pd.read_csv('/home/sachingpt/ML-Project/notebook/StudentsPerformance.csv')

In [ ]:
data.head()

In [ ]:
X=data.drop('math score',axis=1)

In [ ]:
y=data['math score']

In [ ]:
#Create column transformers with 3 types of transformers
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
numeric_features =X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    transformers=[
        ('OneHotEncoder', categorical_transformer, categorical_features),
        ('StandardScaler', numeric_transformer, numeric_features),
    ])

In [ ]:
X=preprocessor.fit_transform(X)

In [ ]:
X.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
def evaluate_model(true,predicted):
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)
    r2 = r2_score(true, predicted)
    mae = mean_absolute_error(true, predicted)
    return rmse, r2, mae


In [ ]:
models={
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'Decision Tree Regressor': DecisionTreeRegressor(),
    'Random Forest Regressor': RandomForestRegressor(),
    'AdaBoost Regressor': AdaBoostRegressor(),
    'Support Vector Regressor': SVR(),
    'CatBoost Regressor': CatBoostRegressor(verbose=0),
    'XGBoost Regressor': XGBRegressor(eval_metric='rmse')
}
model_list = []
r2_list = []

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    model_train_mae, model_train_r2, model_train_rmse = evaluate_model(y_train, y_train_pred)
    model_test_mae, model_test_r2, model_test_rmse = evaluate_model(y_test, y_test_pred)

    print(list(models.keys())[i])
    model_list.append(list(models.keys())[i])

    print('Model Performance on Train Data:')
    print(f'MAE: {model_train_mae}, R2: {model_train_r2}, RMSE: {model_train_rmse}')
    print('Model Performance on Test Data:')
    print(f'MAE: {model_test_mae}, R2: {model_test_r2}, RMSE: {model_test_rmse}')
    print('-----------------------------------')
    r2_list.append(model_test_r2)

    print('='*50)
    print('\n')

    

In [ ]:
pd.DataFrame({'Model': model_list, 'R2 Score': r2_list}).sort_values(by='R2 Score', ascending=False)

In [ ]:
#We are going with Linear Regression as it has the highest R2 score
linear_model = LinearRegression(fit_intercept=True)
linear_model.fit(X_train, y_train)
y_pred=linear_model.predict(X_test)
score=r2_score(y_test, y_pred)*100
print(f'Accuracy of Linear Regression Model: {score:.2f}%')

In [ ]:
#Plot y_pred and y_test
plt.scatter(y_test, y_pred, color='blue', alpha=0.5)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Actual vs Predicted Values')

In [ ]:
sns.regplot(x=y_test, y=y_pred, scatter_kws={'alpha':0.5}, line_kws={'color':'red'})

In [ ]:
#Difference between y_test and y_pred
diff =pd.DataFrame({'y_test': y_test, 'y_pred': y_pred})
diff['Difference'] = diff['y_test'] - diff['y_pred']

In [ ]:
print(diff.head())